In [2]:
from ipaddress import summarize_address_range

import dotenv
from langchain.evaluation.qa.eval_prompt import cot_template

dotenv.load_dotenv()

True

In [3]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader('./input/The_Adventures_of_Tom_Sawyer.pdf')
document = loader.load()
document[5].page_content[:5000]

'Chapter 1    The Fence \n \nTom Sawyer lived with his aunt because his mother and \nfather were dead. Tom didn’t like going to school, and he \ndidn’t like working. He liked playing and having \nadventures. One Friday, he didn’t go to school—he went \nto the river. \nAunt Polly was angry. “You’re a bad boy!” she said. \n“Tomorrow you can’t play with your friends because you \ndidn’t go to school today. Tomorrow you’re going to work \nfor me. You can paint the fence.” \nSaturday morning, Tom was not happy, but he started to \npaint the fence. His friend Jim was in the street. \nTom asked him, “Do you want to paint?” \nJim said, “No, I can’t. I’m going to get water.” \nThen Ben came to Tom’s house. He watched Tom and \nsaid, “I’m going to swim today. You can’t swim because \nyou’re working.” \nTom said, “This isn’t work. I like painting.” \n“Can I paint, too?” Ben asked. \n“No, you can’t,” Tom answered. “Aunt Polly asked me \nbecause I’m a very good painter.” \nBen said, “I’m a good pai

In [4]:
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings() # 임베딩 처리
db = FAISS.from_documents(document, embeddings) # 적재

In [6]:
text = "진희는 강아지를 키우고 있습니다. 진희가 키우고 있는 동물은?"
text_embedding = embeddings.embed_query(text)
print(text_embedding)

[-0.0028138665948063135, -0.020238375291228294, -0.012738940306007862, -0.016539160162210464, -0.02259930968284607, 0.028709961101412773, -0.02787669003009796, 0.00634106295183301, -0.012025609612464905, 0.009797242470085621, -0.015478633344173431, 0.014367605559527874, 0.006716666277498007, -0.033381324261426926, 0.0005783971282653511, 0.028053443878889084, 0.02227105014026165, -0.005046968813985586, 0.02413959801197052, -0.02709392085671425, -0.0003393052611500025, 0.0038601893465965986, 0.014026721939444542, -0.02090751752257347, -0.005157439969480038, 0.018306702375411987, 0.02432897686958313, -0.020415131002664566, -0.00416319677606225, -0.01867283694446087, 0.001243593287654221, -0.0012057173298671842, -0.03547712787985802, -0.005072219297289848, 0.0012278116773813963, -0.017713313922286034, 0.0110850241035223, 0.0072658671997487545, 0.009576299227774143, 0.00325101800262928, -0.0006107494700700045, 0.003241549013182521, 0.00542888417840004, -0.007714065723121166, 0.0155796352773

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

text = "진희는 강아지를 키우고 있습니다. 진희가 키우고 있는 동물은?"
text_embedding = embeddings.embed_query(text)
print(text_embedding)

[0.013278793543577194, 0.07225918024778366, 0.0926310271024704, -0.00397957768291235, 0.0015618291217833757, -0.10306371748447418, 0.10929882526397705, 0.05566202849149704, -0.031167369335889816, -0.05020315572619438, 0.08312956988811493, -0.008924387395381927, 0.0950632318854332, -0.06980786472558975, 0.03955903649330139, -0.10899198800325394, 0.049438633024692535, 0.037364810705184937, -0.12409229576587677, -0.003315416630357504, 0.04840952530503273, -0.031085100024938583, 0.00820700079202652, 0.06326043605804443, -0.06804247945547104, -0.01020818017423153, 0.004927084781229496, -0.014940343797206879, -0.0014765729429200292, -0.006598915904760361, -0.040159497410058975, 0.0828980877995491, 0.014144661836326122, -0.011793515644967556, -0.09415140002965927, 0.0021563717164099216, -0.019053103402256966, -0.03773897886276245, -0.003271071007475257, 0.04685605689883232, -0.1811162233352661, -0.11718792468309402, 0.035048436373472214, -0.06848114728927612, 0.06553441286087036, 0.0352285765

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0,
                 model_name='gpt-4o-mini-2024-07-18')

from langchain.chains import RetrievalQA
retriever = db.as_retriever()

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever
)

query = '마을 무덤에 있던 남자를 죽인 사람은 누구니?'
result = qa.invoke({'query': query})
print(result['result'])

무덤에서 남자를 죽인 사람은 인준 조(Injun Joe)입니다.


In [11]:
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate

llm = ChatOpenAI(temperature=0,
                 model_name='gpt-4o-mini')

prompt = PromptTemplate(
    input_variables=['country'],
    template='{country}의 수도는 어디야?'
)

chain = LLMChain(llm=llm, prompt=prompt) # 프롬프트와 모델을 체인으로 연결
response = chain.invoke('대한민국')
response['text']

'대한민국의 수도는 서울입니다.'

In [12]:
# 프롬프트1 정의
prompt1 = PromptTemplate(
    input_variables=['sentence'],
    template='다음 문장을 한글로 번역하세요.\n\n{sentence}'
)
# 번역 (체인1)에 대한 모델
chain1 = LLMChain(llm=llm, prompt=prompt1, output_key='translation')

In [14]:
# 프롬프트2 정의
prompt2 = PromptTemplate.from_template(
    '다음 문장을 한 문장으로 요약하세요.\n\n{translation}'
)

# 요약 (체인2)에 대한 모델
chain2 = LLMChain(llm=llm, prompt=prompt2, output_key='summary')

In [15]:
from langchain.chains import SequentialChain
all_chain = SequentialChain(
    chains=[chain1, chain2],
    input_variables=['sentence'],
    output_variables=['translation', 'summary']
)

# 번역하고 요약해야 할 영어 문장
sentence = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
response = all_chain.invoke(sentence)

In [16]:
print(f"translation: {response['translation']}")
print()
print(f"summary: {response['summary']}")

translation: LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보가 부족하다는 점입니다. 이를 해결하기 위해 LLM에 특정 외부 데이터에 대한 접근 권한을 부여할 수 있습니다. 이를 위해 먼저 문서 로더를 사용하여 외부 데이터를 로드해야 합니다. LangChain은 PDF, 이메일, 웹사이트, YouTube 비디오 등 다양한 유형의 문서에 대한 다양한 로더를 제공합니다.

summary: LLM의 맥락 정보 부족 문제를 해결하기 위해 LangChain의 다양한 문서 로더를 사용하여 외부 데이터에 접근할 수 있습니다.


In [18]:
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description='ingredient of the dish')
    steps: list[str] = Field(description='steps to make the dish')

output_parser = PydanticOutputParser(pydantic_object=Recipe) # output_parser

template = """다음 요리의 레시피를 생각해 주세요.

{format_instructions}

요리: {dish}
"""

# PromptTemplate
prompt = PromptTemplate(
    template=template,
    input_variables=['dish'],
    partial_variables={'format_instructions': output_parser.get_format_instructions()}
)

chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0) # language model

In [19]:
from langchain.chains import LLMChain

chain = LLMChain(llm=chat, prompt=prompt, output_parser=output_parser)

recipe = chain.invoke('카레')

print(type(recipe))
print(recipe)

<class 'dict'>
{'dish': '카레', 'text': Recipe(ingredients=['1 tablespoon vegetable oil', '1 onion, chopped', '2 cloves garlic, minced', '1 tablespoon ginger, grated', '2 carrots, diced', '1 bell pepper, diced', '2 potatoes, diced', '1 can (400g) diced tomatoes', '2 tablespoons curry powder', '1 teaspoon cumin', '1 teaspoon turmeric', '1 can (400ml) coconut milk', 'Salt and pepper to taste', 'Fresh cilantro for garnish'], steps=['Heat the vegetable oil in a large pot over medium heat.', 'Add the chopped onion and sauté until translucent.', 'Stir in the minced garlic and grated ginger, cooking for another minute.', 'Add the diced carrots, bell pepper, and potatoes, and cook for about 5 minutes.', 'Stir in the diced tomatoes, curry powder, cumin, and turmeric, mixing well.', 'Pour in the coconut milk and bring the mixture to a simmer.', 'Reduce the heat and let it cook for about 20-25 minutes, or until the vegetables are tender.', 'Season with salt and pepper to taste.', 'Serve hot, garnis

In [20]:
chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

cot_template = """다음 질문에 답하세요.

질문: {question}

단계별로 생각해 봅시다.
"""

cot_prompt = PromptTemplate(
    input_variables=['question'],
    template=cot_template
)

cot_chain = LLMChain(llm=chat, prompt=cot_prompt)

In [21]:
summarize_template = """다음 문장을 결론만 간단히 요약하세요.

{input}
"""

summarize_prompt = PromptTemplate(
    input_variables=['input'],
    template=summarize_template
)

summary_chain = LLMChain(llm=chat, prompt=summarize_prompt)

In [22]:
from langchain.chains import SimpleSequentialChain

cot_summarize_chain = SimpleSequentialChain(chains=[cot_chain, summary_chain])

result = cot_summarize_chain.invoke(
    '저는 시작에 가서 사과 10개를 샀습니다. 이웃에게 2개, 수리공에서 2개를 주었습니다. 그런 다음에 사과 5개를 더 사서 1개를 먹었습니다. 남은 개수는 몇 개인가요?'
)
print(result['output'])

남은 사과의 개수는 **10개**입니다.


In [23]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

from langchain import ConversationChain
conversation = ConversationChain(llm=llm, verbose=True)

conversation.predict(input='진희는 강아지를 한마리 키우고 있습니다.')
conversation.predict(input='영수는 고양이를 두마리 키우고 있습니다.')
conversation.predict(input='진희와 영수가 키우는 동물은 총 몇마리?')

C:\Users\kui45\AppData\Local\Temp\ipykernel_3516\4072194998.py:5: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  conversation = ConversationChain(llm=llm, verbose=True)
C:\Users\kui45\anaconda3\envs\PythonProject_rag_2504\lib\site-packages\pydantic\main.py:253: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 진희는 강아지를 한마리 키우고 있습니다.
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: 진희는 강아지를 한마리 키우고 있습니다.
AI: 아, 진희가 강아지를 키우고 있다니 정말 귀엽네요! 어떤 종류의 강아지를 키우고 있는지 궁금해요. 강아지의 이름은 무엇인가요? 그리고 진희는 강아지와 함께 어떤 활동을 즐기나요? 산책이나 놀이, 훈련 같은 것들이요!
Human: 영수는 고양이를 두마리 키우고 있습니다.
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a frie

'진희는 강아지를 한 마리 키우고 있고, 영수는 고양이를 두 마리 키우고 있으니, 총 동물의 수는 1 + 2 = 3마리입니다! 진희의 강아지와 영수의 고양이들이 함께하는 모습이 상상만 해도 재미있네요!'

In [27]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)
conversation = ConversationChain(llm=chat,
                                 memory=ConversationBufferMemory()
                                 )

while True:
    user_message = input('You: ')

    if user_message == '끝':
        print('(대화 종료)')
        break

    ai_message = conversation.invoke(input=user_message)['response']
    print(f'AI: {ai_message}')

AI: 안녕하세요! 어떻게 지내세요? 오늘은 어떤 이야기를 나눠볼까요?
AI: 여행 추천이라니, 정말 흥미로운 주제네요! 어떤 종류의 여행을 원하시나요? 예를 들어, 자연을 즐길 수 있는 곳, 역사적인 명소가 많은 도시, 혹은 해변에서 휴식을 취할 수 있는 장소 등 여러 가지가 있어요. 

1. **자연 여행**: 만약 자연을 좋아하신다면, 제주도는 정말 멋진 선택이에요. 아름다운 해변과 한라산, 그리고 독특한 용암 동굴들이 있어요. 

2. **역사 여행**: 역사에 관심이 많으시다면, 경주를 추천해 드려요. 신라의 고도인 경주는 불국사와 석굴암 같은 유네스코 세계문화유산이 있어요.

3. **해변 휴양**: 해변에서의 휴식을 원하신다면, 부산의 해운대나 강릉의 경포대도 좋습니다. 여름에는 해수욕을 즐길 수 있고, 맛있는 해산물도 많이 있어요.

어떤 여행 스타일이 가장 끌리시나요?
(대화 종료)


In [30]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType

tools = load_tools(['wikipedia', 'llm-math'], llm=llm) # llm-math 같은 경우 나이 계산을 위해 사용
agent = initialize_agent(tools,
                         llm,
                         agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                         description='계산이 필요할 때 사용',
                         verbose=True
)

agent.run('에드 시런이 태어난 해는? 2025년도 현재 에드 시런은 몇 살?')



> Entering new AgentExecutor chain...
에드 시런의 출생 연도를 알아야 그의 나이를 계산할 수 있습니다. 먼저 에드 시런의 출생 연도를 찾기 위해 위키피디아를 검색하겠습니다.  
Action: wikipedia  
Action Input: "Ed Sheeran"  
Observation: Page: Ed Sheeran
Summary: Edward Christopher Sheeran ( SHEER-ən; born 17 February 1991) is an English singer-songwriter. Born in Halifax, West Yorkshire, and raised in Framlingham, Suffolk, he began writing songs around the age of eleven. In early 2011, Sheeran independently released the extended play No. 5 Collaborations Project. He signed with Asylum Records the same year.
Sheeran's debut album, + ("Plus"), was released in September 2011 and topped the UK Albums Chart. It contained his first hit single, "The A Team". In 2012, Sheeran won the Brit Awards for Best British Male Solo Artist and British Breakthrough Act. Sheeran's second studio album, × ("Multiply"), topped charts around the world upon its release in June 2014. It was named the second-best-selling album worldwide of 2015. In the same year, × won

'에드 시런은 1991년 2월 17일에 태어났으며, 2025년 현재 34세입니다.'